# Programming Assignment 29: Temporal Difference Learning (Student Performance)

## Objective
Predict long-term student performance using Temporal Difference (TD) Learning on the UCI Student Performance Dataset.

## Dataset
**UCI Student Performance Dataset**
We use `student-mat.csv` (Math course).
State: Discretized grades (0-5 scale).
Reward: Improvement between terms.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import zipfile
import io
import os

In [ ]:
# -----------------------------
# 1) LOAD REAL DATASET
# -----------------------------
# Check if file exists, if not download from UCI repository
file_path = "student-mat.csv"

if not os.path.exists(file_path):
    print("Dataset not found. Downloading from UCI repository...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
    try:
        r = requests.get(url)
        r.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extract("student-mat.csv")
        print("Download and extraction complete.")
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        print("Please manually download student-mat.csv from UCI Machine Learning Repository.")
else:
    print("Dataset found locally.")

# Load data
if os.path.exists(file_path):
    df = pd.read_csv(file_path, sep=";")
    print("Dataset loaded. Shape:", df.shape)

    # We use G1 (term1), G2 (term2), G3 (final)
    grades = df[["G1", "G2", "G3"]].copy()
else:
    print("Could not run assignment without dataset.")

In [ ]:
# -----------------------------
# 2) DISCRETIZE GRADES → STATES
# -----------------------------
def grade_to_state(g):
    if g <= 5: return 0
    elif g <= 9: return 1
    elif g <= 12: return 2
    elif g <= 15: return 3
    elif g <= 18: return 4
    else: return 5

# Using .apply and creating new columns directly on the dataframe copy
if 'grades' in locals():
    grades["S1"] = grades["G1"].apply(grade_to_state)
    grades["S2"] = grades["G2"].apply(grade_to_state)
    grades["S3"] = grades["G3"].apply(grade_to_state)

    print(grades[["G1", "S1", "G2", "S2", "G3", "S3"]].head())

In [ ]:
# -----------------------------
# 3) DEFINE MRP TRAJECTORIES
# -----------------------------
# Each student = one episode
episodes = []

if 'grades' in locals():
    for _, row in grades.iterrows():
        states = [row["S1"], row["S2"], row["S3"]]
        # Reward structure: Improvement between terms, and Final Grade as terminal reward
        rewards = [row["G2"] - row["G1"], row["G3"] - row["G2"], row["G3"]]
        episodes.append((states, rewards))

    print(f"Created {len(episodes)} episodes.")
    if len(episodes) > 0:
        print("Example trajectory:", episodes[0])

In [ ]:
# -----------------------------
# 4) TD(0) LEARNING
# -----------------------------
S = 6  # States 0 to 5
gamma = 0.9
alpha = 0.1
V = np.zeros(S)

def td0_update(states, rewards):
    global V
    # Standard TD updates for transitions
    for t in range(len(states)-1):
        s, s_next = states[t], states[t+1]
        r = rewards[t]
        V[s] += alpha * (r + gamma * V[s_next] - V[s])
    
    # Terminal update
    # In this formulation, the final state 's_T' gets updated towards the final reward
    V[states[-1]] += alpha * (rewards[-1] - V[states[-1]])

# Training
if len(episodes) > 0:
    for _ in range(50):  # multiple passes over dataset
        for states, rewards in episodes:
            td0_update(states, rewards)
            
    print("Training complete.")

In [ ]:
# -----------------------------
# 5) VISUALIZATION
# -----------------------------
state_labels = [
    "Very Poor (<=5)", "Poor (6-9)", "Average (10-12)",
    "Good (13-15)", "Very Good (16-18)", "Excellent (>18)"
]

plt.figure(figsize=(10, 6))
plt.bar(state_labels, V, color='cornflowerblue', edgecolor='black')
plt.title("TD(0) Predicted Long-Term Student Performance")
plt.ylabel("Expected Long-Term Reward")
plt.xlabel("Current Grade State")
plt.xticks(rotation=30)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

In [ ]:
# -----------------------------
# 6) OUTPUT
# -----------------------------
print("Learned Value Function V(s):")
for i, v in enumerate(V):
    print(f"State {i} ({state_labels[i]}): {v:.2f}")